# Site Reliability Server - GRPO Training
This notebook starts the OpenEnv FastAPI server, runs a minimal TRL GRPO job, exports local metrics, and writes submission plots.

In [1]:
repo_url = "https://github.com/siddheshkadane01/Site-Reliability-Server/"
!git clone {repo_url}
repo_dir = repo_url.rstrip('/').split('/')[-1].replace('.git', '')
%cd $repo_dir

Cloning into 'Site-Reliability-Server'...
remote: Enumerating objects: 339, done.
remote: Counting objects: 100% (339/339), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 339 (delta 225), reused 334 (delta 222), pack-reused 0 (from 0)
Receiving objects: 100% (339/339), 961.73 KiB | 1.55 MiB/s, done.
Resolving deltas: 100% (225/225), done.
Filtering content: 100% (2/2), 3.40 KiB | 1024 bytes/s, done.
/Users/sriram_kommalapudi/Projects/Site-Reliability-Server/Site-Reliability-Server


In [ ]:
%pip install -q -r requirements.txt
%pip install -q -r requirements_rl.txt

In [ ]:
import subprocess, time, requests, pathlib

# Ensure all task scenarios exist in fresh Colab clones.
required = ["easy", "medium", "hard", "expert", "enterprise"]
missing = [
    t for t in required
    if not (pathlib.Path("scenarios") / t).exists()
    or not list((pathlib.Path("scenarios") / t).glob("*.json"))
]
if missing:
    print(f"Generating missing scenarios for: {missing}")
    subprocess.run(["python", "env/data_generator.py"], check=True)

server_log = open('server.log', 'w')
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '7860'],
    stdout=server_log,
    stderr=server_log,
)
for _ in range(30):
    try:
        print(requests.get('http://127.0.0.1:7860/health', timeout=2).json())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(open('server.log').read())

In [1]:
!WANDB_MODE=disabled python train_grpo.py --epochs 1 --samples 16 --task_id enterprise --model_name Qwen/Qwen2.5-0.5B-Instruct

/Users/sriram_kommalapudi/Projects/Site-Reliability-Server/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sriram_kommalapudi/Projects/Site-Reliability-Server/train_grpo.py:20: FutureWarning: Support for Python 3.9 will be dropped in the next release (after its end-of-life on October 31, 2025). Please upgrade to Python 3.10 or newer.
  from trl import GRPOConfig, GRPOTrainer
`torch_dtype` is deprecated! Use `dtype` instead!
Traceback (most recent call last):
  File "/Users/sriram_kommalapudi/Projects/Site-Reliability-Server/train_grpo.py", line 291, in <module>
    train(build_parser().parse_args())
  File "/Users/sriram_kommalapudi/Projects/Site-Reliability-Server/train_grpo.py", line 220, in train
    config = GRPOConfig(
  File "<string>", line 183, in __init__
  File "/Users/sr

In [ ]:
!python scripts/plot_training_artifacts.py --metrics artifacts/training_metrics.jsonl --output-dir artifacts/submission
from IPython.display import display
display({"image/png": open("artifacts/submission/reward_curve.png", "rb").read()}, raw=True)
display({"image/png": open("artifacts/submission/loss_curve.png", "rb").read()}, raw=True)